In [1]:
import boto3
import json
from IPython.display import JSON

In [2]:
bedrock = boto3.client(service_name="bedrock-agent", region_name="us-east-1")
MODEL_ID = "amazon.nova-lite-v1:0"
MODEL_ID_GPT_OSS_120B= "openai.gpt-oss-120b-1:0"

In [3]:
try:
    response = bedrock.create_prompt(
        name="transcriptEvaluator-ghi",
        description="Avaliador de transcrições",
        variants=[{
            "name": "Variant1",
            "modelId": MODEL_ID,
            "templateType": "CHAT",
            "inferenceConfiguration": {
                "text": {
                    "temperature": 0.8
                }
            },
            "templateConfiguration": {
                "chat": {
                    "system": [
                        {
                            "text": """Responda as perguntas abaixo sobre a transcrição apenas com sim ou não. Forneça somente as respostas, uma para cada pergunta.
                            Se não tiver dados suficientes para responder a uma pergunta, responda não.

                            A) O atendente ofereceu a promoção de pacote vigente?
                            B) O cliente aceitou a promoção de pacote?
                            C) O atendente conseguiu responder todas as perguntas do cliente?
                            D) O cliente conseguiu concluir sua solicitação?
                            E) O atendente perguntou se poderia ajudar com mais alguma coisa?"""
                        }
                    ],
                    "messages": [{
                        'role': "user",
                        "content": [
                            {
                                "text": "Transcrição da ligação: {{transcript}}" ## << NUNCA use f-string aqui, cometi esse erro e passei 30min depurando kkk
                            } 
                        ]
                    }],
                    "inputVariables": [
                        {"name": "transcript"}
                    ]
                }
            }
        }]
    )
    print("Criado!")
    prompt_arn = response.get("arn")

except bedrock.exceptions.ConflictException as e:
    print("Já existe!")
    response = bedrock.list_prompts()
    prompt = next((prompt for prompt in response['promptSummaries'] if prompt['name'] == "TranscriptEvaluator_ghi"), None)
    prompt_arn = prompt['arn']

prompt_arn


Criado!


'arn:aws:bedrock:us-east-1:<ACCOUNT_ID>:prompt/199ECC4F8A'

In [4]:
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")

response = bedrock_runtime.converse(
    modelId=prompt_arn,
    promptVariables={
        'transcript': {
            "text": """
            Cliente: Eu gostaria de aumentar a velocidade da minha internet a cabo.
            Atendente: Sim, podemos fazer isso. Hoje o senhor tem 512kb de velocidade e podemos aumentar para 1 gigabit. No momento há uma promoção que inclui o canal de esportes no pacote. Tem interesse?
            Cliente: Não, só a velocidade da internet. Quanto custaria 1 gigabit por mês?
            Atendente: Seriam US$ 10 a mais por mês, ou US$ 5 se o senhor fizer fidelidade de 12 meses nesse plano.
            Cliente: Certo, vou ficar com o plano de US$ 5 a mais. Sabe quando passa a valer?
            Atendente: Sim, a nova velocidade estará ativa por volta da meia-noite de hoje (horário da costa leste dos EUA).
            Cliente: Ótimo, obrigado pela ajuda.
            Atendente: Fico feliz em ajudar. Posso ajudar com mais alguma coisa hoje?
            Cliente: Não, era só isso por hoje. Tchau.
            """
        }
    }
)

print(response['output']['message']['content'][0]['text'])

A) Não
B) Não
C) Sim
D) Sim
E) Sim


In [5]:
body_json = json.dumps({"promptVariables": {
        'transcript': {
            "text": """
            Cliente: Eu gostaria de aumentar a velocidade da minha internet a cabo.
            Atendente: Sim, podemos fazer isso. Hoje o senhor tem 512kb de velocidade e podemos aumentar para 1 gigabit. No momento há uma promoção que inclui o canal de esportes no pacote. Tem interesse?
            Cliente: Não, só a velocidade da internet. Quanto custaria 1 gigabit por mês?
            Atendente: Seriam US$ 10 a mais por mês, ou US$ 5 se o senhor fizer fidelidade de 12 meses nesse plano.
            Cliente: Certo, vou ficar com o plano de US$ 5 a mais. Sabe quando passa a valer?
            Atendente: Sim, a nova velocidade estará ativa por volta da meia-noite de hoje (horário da costa leste dos EUA).
            Cliente: Ótimo, obrigado pela ajuda.
            Atendente: Fico feliz em ajudar. Posso ajudar com mais alguma coisa hoje?
            Cliente: Não, era só isso por hoje. Tchau.
            """
        }
    }})

response = bedrock_runtime.invoke_model(
    body=body_json,
    modelId=prompt_arn
)

json_response = json.loads(response['body'].read())
json_response

{'output': {'message': {'content': [{'text': 'A) Não  \nB) Não  \nC) Sim  \nD) Sim  \nE) Sim'}],
   'role': 'assistant'}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 356,
  'outputTokens': 20,
  'totalTokens': 376,
  'cacheReadInputTokenCount': 0,
  'cacheWriteInputTokenCount': 0}}